In [ ]:
import os.path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import csv
from collections import defaultdict
import re

In [ ]:
file_name = '2026-01-21_6_not-separate-schema-build_latency.csv'

all_data = []


df = pd.read_csv(os.path.join('./result/metrics/',file_name))
provModels = df['provModel'].unique().tolist()
datasets = df['dataset'].unique().tolist()

# provModels = df['provModel'].unique().tolist()
# datasets = df['dataset'].unique().tolist()

for provModel in provModels:
    all_data = {
    }
    for dataset in datasets:
        scale_factors = [0.1]
        # scale_factors = df['scaleFactor'].unique().tolist()
        sf_data=[]
        for sf in scale_factors:
            dt_sf_prov = df[ (df['provModel']== provModel) & (df['dataset'] == dataset) & (df['scaleFactor'] == sf)][['query','mean']]

            orig = dt_sf_prov[dt_sf_prov['query'].str.startswith('orig_')].set_index('query')
            prov = dt_sf_prov[dt_sf_prov['query'].str.startswith('prov_')].set_index('query')

            orig.index = orig.index.str.replace('orig_', '')
            prov.index = prov.index.str.replace('prov_', '')

            comparison = orig.join(prov, lsuffix='_orig', rsuffix='_prov')
            sf_data.append((sf,comparison))
        all_data[dataset]=sf_data

    fig, ax = plt.subplots(figsize=(14, 6))

    bar_width = 0.35
    gap_queries = 0.2
    gap_sf = 1.0
    gap_dataset = 2.0

    x_positions = []
    query_labels = []

    current_x = 0

    sf_centers = []
    dataset_centers = []

    for dataset_name, sf_list in all_data.items():
        dataset_start = current_x

        for sf_idx, (sf, comparison) in enumerate(sf_list):
            n_queries = len(comparison)
            x_inner = np.arange(current_x, current_x + n_queries)

            # Store query ticks
            x_positions.extend(x_inner)
            query_labels.extend(comparison.index)

            # Plot bars
            ax.bar(x_inner - bar_width / 2, comparison['mean_orig'],
                   width=bar_width,
                   label='orig' if current_x == 0 else "")
            ax.bar(x_inner + bar_width / 2, comparison['mean_prov'],
                   width=bar_width,
                   label='prov' if current_x == 0 else "")

            # Scale factor center
            sf_centers.append((x_inner.mean(), sf))

            current_x += n_queries + gap_sf

        dataset_end = current_x - gap_sf
        dataset_centers.append(((dataset_start + dataset_end) / 2, dataset_name))
        current_x += gap_dataset

    # ---- Query ticks (level 1)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(query_labels, rotation=45, ha='right')

    ax.set_ylabel('Mean Value')
    ax.set_title(f"{provModel} Provenance : Original vs Provenance Query Latencies")

    # ---- Scale factor labels (level 2)
    ymin, ymax = ax.get_ylim()
    sf_y = ymin - (ymax - ymin) * 0.08

    for x, sf in sf_centers:
        ax.text(x, sf_y, f'SF {sf}', ha='center', va='top', fontsize=10)

    # ---- Dataset labels (level 3)
    dataset_y = ymin - (ymax - ymin) * 0.15

    for x, name in dataset_centers:
        ax.text(x, dataset_y, name, ha='center', va='top',
                fontsize=11, fontweight='bold')

    # ---- Layout
    plt.subplots_adjust(bottom=0.30)
    ax.legend()
    plt.show()




In [ ]:

file_name = '2026-01-21_1_latency.csv'

def query_number(qid):
    """
    Extract numeric suffix from query id, e.g. finbench-15 -> 15
    """
    m = re.search(r"-(\d+(?:\.\d+)?)$", qid)
    return float(m.group(1)) if m else float("inf")


def make_tables(filename):
    groups = defaultdict(lambda: defaultdict(dict))

    with open(os.path.join('./result/metrics/',file_name), newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            dataset = row["dataset"]
            scale = row["scaleFactor"]
            prov_model = row["provModel"]
            query = row["query"]
            mean = float(row["mean"])
            if "hitRate" in row.keys():
                hit_rate = float(row["hitRate"])
            else:
                hit_rate = "N/A"

            if query.startswith("orig_"):
                qid = query.replace("orig_", "")
                groups[(dataset, scale, prov_model)][qid]["orig"] = mean
                groups[(dataset, scale, prov_model)][qid]["hit_rate"] = hit_rate
            elif query.startswith("prov_"):
                qid = query.replace("prov_", "")
                groups[(dataset, scale, prov_model)][qid]["prov"] = mean
                groups[(dataset, scale, prov_model)][qid]["hit_rate"] = hit_rate

    # Print tables sorted by query number
    for (dataset, scale, prov_model), queries in sorted(groups.items()):
        print(f"\nDataset: {dataset}, ScaleFactor: {scale}, ProvModel: {prov_model}")
        print(f"{'Query':<15} {'Orig Mean':>12} {'Prov Mean':>12} {'Hit Rate':>12} {'% Change':>12}")
        print("-" * 60)

        for qid in sorted(queries.keys(), key=query_number):
            vals = queries[qid]
            orig = vals.get("orig")
            prov = vals.get("prov")
            hitRate = vals.get("hit_rate")

            if orig is not None and prov is not None:
                pct = ((prov - orig) / orig) * 100
                pct_str = f"{pct:.2f}%"
                orig_str = f"{orig:.4f}"
                prov_str = f"{prov:.4f}"
            else:
                pct_str = "N/A"
                orig_str = f"{orig:.4f}" if orig is not None else "N/A"
                prov_str = f"{prov:.4f}" if prov is not None else "N/A"

            print(f"{qid:<15} {orig_str:>12} {prov_str:>12} {hitRate:>12} {pct_str:>12}")

make_tables(file_name)

In [ ]:
file_name = '2026-01-21_4_latency.csv'
make_tables(file_name)

In [ ]:
file_name='2026-01-21_5_not-separate-schema-build_latency.csv'
make_tables(file_name)

In [ ]:
file_name='2026-01-21_6_not-separate-schema-build_latency.csv'
make_tables(file_name)

In [ ]:
file_name='2026-01-21_6.2_not-separate-schema-build_latency.csv'
make_tables(file_name)

In [ ]:
def plot_experiment_results_from_tuples(data):
    """
    data: list of tuples (scale_factor, file_path)
    """

    all_data = []
    scale_factors = []

    for sf, file_path in data:
        scale_factors.append(sf)
        df = pd.read_csv(file_path)

        # Split into orig and prov
        orig = df[df['query'].str.startswith('orig_')].set_index('query')
        prov = df[df['query'].str.startswith('prov_')].set_index('query')
